# Laboratorio 01.2 — Muestreo y generación

Estudiaremos cómo temperatura y nucleus sampling transforman una distribución de logits. Usaremos una simulación reproducible y local antes de conectar cualquier modelo real.

## 1. Objetivos de aprendizaje

Al finalizar podrás:
- Convertir logits en probabilidades mediante softmax.
- Explicar el efecto de temperatura en la concentración de una distribución.
- Implementar top-p conservando el conjunto mínimo de masa acumulada.
- Medir entropía y variación muestral con una semilla fija.

## 2. Conceptos

Un modelo asigna un logit $z_i$ a cada token candidato. Softmax convierte los logits en probabilidades. La temperatura $T$ escala los logits antes de normalizar: bajar $T$ concentra masa en las opciones probables; subirla la distribuye. Top-p conserva los tokens con mayor probabilidad hasta alcanzar una masa acumulada $p$. Son controles de aleatoriedad, no medidas directas de calidad ni de veracidad.

## 3. Arquitectura

```mermaid
flowchart LR
    A[Logits] --> B[Escalado por temperatura]
    B --> C[Softmax]
    C --> D[Filtro top-p]
    D --> E[Muestreo con semilla]
    E --> F[Frecuencias y entropia]
```

## 4. Preparación del entorno

Solo necesitamos NumPy para las distribuciones y pandas/matplotlib para resumir y visualizar resultados. El laboratorio no llama a un proveedor.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from llm_engineering.environment import preparar_entorno

raiz_curso = preparar_entorno()
generador = np.random.default_rng(17)

## 5. Definición del problema

La siguiente distribución representa los logits de cinco continuaciones posibles en un paso de generación. Nos interesa medir cómo cambian concentración y diversidad cuando se modifica un solo parámetro, manteniendo fijos logits y semilla.

## 6. Línea base

Argmax siempre selecciona la opción de mayor logit. Es determinista y evita variación de muestreo, pero también ignora toda la masa asignada a alternativas plausibles.

In [ ]:
tokens_candidatos = ['respuesta', 'explicacion', 'ejemplo', 'pregunta', 'desconocido']
logits = np.array([2.4, 1.7, 0.8, 0.1, -0.6], dtype=float)
token_argmax = tokens_candidatos[int(np.argmax(logits))]
print('Continuacion de linea base:', token_argmax)

## 7. Implementación

Implementamos softmax estable restando el logit máximo. Luego aplicamos top-p en orden descendente y renormalizamos la masa conservada. La semilla hace repetible la secuencia de muestras, no convierte al muestreo en una respuesta universalmente correcta.

In [ ]:
def softmax_con_temperatura(valores: np.ndarray, temperatura: float) -> np.ndarray:
    """Devuelve probabilidades normalizadas para una temperatura positiva."""
    if temperatura <= 0:
        raise ValueError('La temperatura debe ser mayor que cero.')
    escalados = valores / temperatura
    exponentes = np.exp(escalados - np.max(escalados))
    return exponentes / exponentes.sum()


def aplicar_top_p(probabilidades: np.ndarray, limite: float) -> np.ndarray:
    """Conserva los tokens mas probables hasta cubrir la masa limite."""
    if not 0 < limite <= 1:
        raise ValueError('top_p debe estar en el intervalo (0, 1].')
    orden = np.argsort(probabilidades)[::-1]
    acumulada = np.cumsum(probabilidades[orden])
    conservar = acumulada - probabilidades[orden] < limite
    conservar[0] = True
    filtradas = np.zeros_like(probabilidades)
    filtradas[orden[conservar]] = probabilidades[orden[conservar]]
    return filtradas / filtradas.sum()


def medir_entropia(probabilidades: np.ndarray) -> float:
    """Calcula entropia de Shannon ignorando probabilidades nulas."""
    positivas = probabilidades[probabilidades > 0]
    return float(-(positivas * np.log2(positivas)).sum())

probabilidades = softmax_con_temperatura(logits, temperatura=1.0)
print(pd.DataFrame({'token': tokens_candidatos, 'probabilidad': probabilidades}))

## 8. Experimentos

En la primera comparación cambia solo la temperatura; en la segunda cambia solo top-p. Para cada configuración muestreamos 2.000 veces con una semilla conocida y registramos distribución y entropía.

In [ ]:
def resumir_muestras(temperatura: float, top_p: float, semilla: int, cantidad: int = 2000) -> dict[str, object]:
    """Resume una configuracion de muestreo reproducible."""
    distribucion = aplicar_top_p(softmax_con_temperatura(logits, temperatura), top_p)
    rng = np.random.default_rng(semilla)
    muestras = rng.choice(tokens_candidatos, size=cantidad, p=distribucion)
    frecuencias = pd.Series(muestras).value_counts(normalize=True)
    return {
        'temperatura': temperatura,
        'top_p': top_p,
        'entropia_bits': medir_entropia(distribucion),
        'tokens_distintos': int(frecuencias.size),
        'token_mas_frecuente': str(frecuencias.index[0]),
        'frecuencia_mayor': float(frecuencias.iloc[0]),
        'distribucion': distribucion,
    }

pruebas_temperatura = [resumir_muestras(valor, 1.0, 17) for valor in (0.4, 1.0, 1.8)]
pruebas_top_p = [resumir_muestras(1.0, valor, 17) for valor in (0.45, 0.75, 1.0)]
tabla_experimentos = pd.DataFrame([
    {clave: valor for clave, valor in fila.items() if clave != 'distribucion'}
    for fila in pruebas_temperatura + pruebas_top_p
])
display(tabla_experimentos)
tabla_experimentos.to_csv(raiz_curso / 'outputs' / '01_02_sampling_experiments.csv', index=False)

## 9. Evaluación

La entropía resume la dispersión de la distribución y el número de tokens distintos resume diversidad observada en la muestra. Ninguna de estas métricas determina si el texto resultante es útil o correcto; aquí evaluamos propiedades estadísticas controladas.

In [ ]:
resumen_temperatura = pd.DataFrame([
    {
        'temperatura': fila['temperatura'],
        'entropia_bits': fila['entropia_bits'],
        'tokens_distintos': fila['tokens_distintos'],
    }
    for fila in pruebas_temperatura
])
resumen_top_p = pd.DataFrame([
    {
        'top_p': fila['top_p'],
        'entropia_bits': fila['entropia_bits'],
        'tokens_distintos': fila['tokens_distintos'],
    }
    for fila in pruebas_top_p
])
display(resumen_temperatura)
display(resumen_top_p)
figura, ejes = plt.subplots(1, 2, figsize=(10, 4))
for fila in pruebas_temperatura:
    ejes[0].plot(tokens_candidatos, fila['distribucion'], marker='o', label=f"T={fila['temperatura']}")
for fila in pruebas_top_p:
    ejes[1].plot(tokens_candidatos, fila['distribucion'], marker='o', label=f"p={fila['top_p']}")
ejes[0].set_title('Cambio de temperatura')
ejes[1].set_title('Cambio de top-p')
for eje in ejes:
    eje.tick_params(axis='x', rotation=35)
    eje.legend()
plt.tight_layout()
plt.show()

## 10. Discusión

Temperaturas bajas concentran probabilidad; top-p elimina la cola menos probable antes de renormalizar. La diversidad observada también depende de la semilla y del número de muestras. En modelos reales, decodificación, filtros del proveedor y el prompt afectan el resultado además de estos parámetros.

## 11. Ejercicios

1. Cambia los logits manteniendo la semilla y explica cómo cambia la entropía.
2. Implementa top-k y compara su soporte con top-p.
3. Repite cada experimento con varias semillas y calcula la variación entre corridas.

## 12. Desafío

Implementa una búsqueda de temperatura que alcance una entropía objetivo para distintos vectores de logits. Registra los casos que no convergen dentro del rango permitido.

## 13. Ideas clave

- Temperatura modifica la forma de la distribución antes de muestrear.
- Top-p limita dinámicamente el conjunto de candidatos.
- Semillas ayudan a reproducir simulaciones, no garantizan reproducir APIs remotas.
- Diversidad estadística no equivale a calidad de respuesta.